## Imports og tilkobling

In [2]:
import sys
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

sys.path.append(os.path.abspath("../../../"))
from src.feature_engineering.forbruksdata import create_consumption_features

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "rawdata"

## Hente CSV fra blob

In [3]:
blob_name = "AntallMålepunktNorgespris.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

## Feature engineering

In [4]:
from src.feature_engineering.norgespris import clean_norgespris_df, split_by_station

df = clean_norgespris_df(df)

station_dfs = split_by_station(df)

df.head()

,transformer_station,timestamp,count_total
0,TIMENES TS,2025-10-01 00:00:00,1946
1,TIMENES TS,2025-10-02 00:00:00,1990
2,TIMENES TS,2025-10-03 00:00:00,2022
3,TIMENES TS,2025-10-04 00:00:00,2040
4,TIMENES TS,2025-10-05 00:00:00,2055


## Lagre i Azure Blob

In [7]:
output_container = "processeddata"

for station_name, station_df in station_dfs.items():
    
    output_blob_name = f"{station_name}_norgespris.csv"
    
    csv_buffer = io.StringIO()
    station_df.to_csv(csv_buffer, index=False)

    blob_client = blob_service_client.get_blob_client(
        container=output_container,
        blob=output_blob_name
    )

    blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)

    print(f"Lastet opp: {output_blob_name}")

Lastet opp: breive_norgespris.csv
Lastet opp: frikstad_norgespris.csv
Lastet opp: hartevatn_norgespris.csv
Lastet opp: timenes_ts_norgespris.csv


# Kvalitetssikring

## Kun en stasjon per fil

In [5]:
for name, df_station in station_dfs.items():
    unique_stations = df_station["transformer_station"].unique()
    
    print(f"\n{name}")
    print("Antall stasjoner:", len(unique_stations))
    print("Stasjon:", unique_stations)


breive
Antall stasjoner: 1
Stasjon: <StringArray>
['BREIVE']
Length: 1, dtype: str

frikstad
Antall stasjoner: 1
Stasjon: <StringArray>
['FRIKSTAD']
Length: 1, dtype: str

hartevatn
Antall stasjoner: 1
Stasjon: <StringArray>
['HARTEVATN']
Length: 1, dtype: str

timenes_ts
Antall stasjoner: 1
Stasjon: <StringArray>
['TIMENES TS']
Length: 1, dtype: str


## Sjekk timestamp format

In [7]:
for name, df_station in station_dfs.items():
    print(f"\n{name}")
    print(df_station["timestamp"].dtype)
    print(df_station["timestamp"].head(3))


breive
str
427    2025-10-01 00:00:00
428    2025-10-02 00:00:00
429    2025-10-03 00:00:00
Name: timestamp, dtype: str

frikstad
str
267    2025-10-01 00:00:00
268    2025-10-02 00:00:00
269    2025-10-03 00:00:00
Name: timestamp, dtype: str

hartevatn
str
151    2025-10-01 00:00:00
152    2025-10-02 00:00:00
153    2025-10-03 00:00:00
Name: timestamp, dtype: str

timenes_ts
str
0    2025-10-01 00:00:00
1    2025-10-02 00:00:00
2    2025-10-03 00:00:00
Name: timestamp, dtype: str


## Sjekk duplikater

In [8]:
for name, df_station in station_dfs.items():
    duplicates = df_station.duplicated(subset=["timestamp", "count_total"]).sum()
    
    print(f"\n{name}")
    print("Duplikater:", duplicates)


breive
Duplikater: 0

frikstad
Duplikater: 0

hartevatn
Duplikater: 0

timenes_ts
Duplikater: 0


## Sjekk manglende verdier

In [9]:
for name, df_station in station_dfs.items():
    print(f"\n{name}")
    print(df_station.isna().sum())


breive
transformer_station    0
timestamp              0
count_total            0
dtype: int64

frikstad
transformer_station    0
timestamp              0
count_total            0
dtype: int64

hartevatn
transformer_station    0
timestamp              0
count_total            0
dtype: int64

timenes_ts
transformer_station    0
timestamp              0
count_total            0
dtype: int64


## Sjekk nullverdier

In [10]:
for name, df_station in station_dfs.items():
    zeros = (df_station["count_total"] == 0).sum()
    
    print(f"\n{name}")
    print("Nullverdier:", zeros)


breive
Nullverdier: 0

frikstad
Nullverdier: 0

hartevatn
Nullverdier: 0

timenes_ts
Nullverdier: 0


## Sjekk hull i tidsserie

In [34]:
import pandas as pd

# Definer perioden
start_date = '2025-11-01'
end_date = '2026-01-31'
full_days = pd.date_range(start=start_date, end=end_date, freq='D')

for name, df_station in station_dfs.items():
    # Normaliser timestamp til dato (00:00:00)
    df_station['date'] = pd.to_datetime(df_station['timestamp']).dt.normalize()
    
    # Hent unike dager
    station_days = df_station['date'].drop_duplicates()
    
    # Sjekk hvilke dager som mangler
    missing_days = full_days.difference(station_days)
    
    if len(missing_days) == 0:
        print(f"{name}: Ingen dager mangler ✅")
    else:
        print(f"{name}: Mangler {len(missing_days)} dag(er): {missing_days.date.tolist()}")

breive: Mangler 29 dag(er): [datetime.date(2025, 11, 9), datetime.date(2025, 11, 12), datetime.date(2025, 11, 15), datetime.date(2025, 11, 16), datetime.date(2025, 11, 18), datetime.date(2025, 12, 3), datetime.date(2025, 12, 4), datetime.date(2025, 12, 10), datetime.date(2025, 12, 17), datetime.date(2025, 12, 22), datetime.date(2025, 12, 24), datetime.date(2025, 12, 27), datetime.date(2025, 12, 30), datetime.date(2025, 12, 31), datetime.date(2026, 1, 1), datetime.date(2026, 1, 3), datetime.date(2026, 1, 10), datetime.date(2026, 1, 11), datetime.date(2026, 1, 12), datetime.date(2026, 1, 13), datetime.date(2026, 1, 15), datetime.date(2026, 1, 18), datetime.date(2026, 1, 19), datetime.date(2026, 1, 21), datetime.date(2026, 1, 22), datetime.date(2026, 1, 23), datetime.date(2026, 1, 24), datetime.date(2026, 1, 28), datetime.date(2026, 1, 31)]
frikstad: Mangler 4 dag(er): [datetime.date(2025, 12, 22), datetime.date(2025, 12, 23), datetime.date(2025, 12, 31), datetime.date(2026, 1, 15)]
harte

## Sjekk fordeling

In [14]:
for name, df_station in station_dfs.items():
    print(f"\n{name}")
    print(df_station["count_total"].describe())


breive
count     111.000000
mean     1076.162162
std        71.700582
min       878.000000
25%      1030.500000
50%      1093.000000
75%      1133.500000
max      1170.000000
Name: count_total, dtype: float64

frikstad
count     160.000000
mean     4842.925000
std       377.747654
min      3675.000000
25%      4618.500000
50%      4941.000000
75%      5145.000000
max      5264.000000
Name: count_total, dtype: float64

hartevatn
count     116.000000
mean     1829.405172
std       101.725460
min      1521.000000
25%      1766.250000
50%      1855.000000
75%      1914.250000
max      1960.000000
Name: count_total, dtype: float64

timenes_ts
count     151.000000
mean     2585.629139
std       223.668165
min      1946.000000
25%      2442.500000
50%      2647.000000
75%      2773.500000
max      2850.000000
Name: count_total, dtype: float64


## Sjekk endring over tid

In [15]:
for name, df_station in station_dfs.items():
    df_station = df_station.sort_values("timestamp")
    df_station["diff"] = df_station["count_total"].diff()

    print(f"\n{name}")
    print(df_station["diff"].describe())


breive
count    110.000000
mean       2.654545
std        2.978320
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       25.000000
Name: diff, dtype: float64

frikstad
count    159.000000
mean       9.993711
std       13.006083
min        1.000000
25%        3.000000
50%        5.000000
75%       12.000000
max      106.000000
Name: diff, dtype: float64

hartevatn
count    115.000000
mean       3.817391
std        4.566435
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       28.000000
Name: diff, dtype: float64

timenes_ts
count    150.000000
mean       6.026667
std        6.651383
min        1.000000
25%        2.000000
50%        3.000000
75%        7.000000
max       44.000000
Name: diff, dtype: float64
